# Fine-tuning experimental do NCM para Python

Este notebook adapta o **Next-Concept Modeling (NCM)** oficial ao laboratorio existente.

Estratégia:
- checkpoint oficial `hazyresearch/ncm-checkpoints/xsum`;
- `MBPP sanitized` como corpus pequeno de exemplos Python;
- tokenizador/VQ-VAE congelado;
- treino somente do gerador;
- batch pequeno e acumulacao de gradiente para uma RTX 3060 Ti de 8 GB.

Por padrao, `RUN_TRAINING=False`. Execute primeiro todas as verificacoes e o smoke test.

In [1]:
from pathlib import Path
import ast
import gc
import os
import random
import sys

import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GB")

torch: 2.6.0+cu124
CUDA disponivel: True
GPU: NVIDIA GeForce RTX 3060 Ti
VRAM total: 7.66 GB


In [2]:
import importlib.util

required = ["omegaconf", "hydra", "datasets", "huggingface_hub", "transformers"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print("Dependencias ausentes:", missing)
    print("Reconstruir a imagem com: docker compose up -d --build")
else:
    print("Dependencias essenciais encontradas.")

Dependencias essenciais encontradas.


In [3]:
import subprocess

NCM_COMMIT = "c70274273f489abc73832b2b98584a8483b6b203"
NCM_DIR = Path("/workspace/models/ncm")

if not (NCM_DIR / ".git").exists():
    NCM_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "https://github.com/HazyResearch/ncm.git", str(NCM_DIR)
    ], check=True)

subprocess.run(["git", "-C", str(NCM_DIR), "checkout", NCM_COMMIT], check=True)
sys.path.insert(0, str(NCM_DIR))
print("NCM:", NCM_DIR)
print("commit:", NCM_COMMIT)

Cloning into '/workspace/models/ncm'...


NCM: /workspace/models/ncm
commit: c70274273f489abc73832b2b98584a8483b6b203


Note: switching to 'c70274273f489abc73832b2b98584a8483b6b203'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at c702742 update bibtex


In [4]:
from huggingface_hub import hf_hub_download

CHECKPOINT_DIR = Path("/workspace/models/ncm-checkpoints/xsum")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

for filename in ("config.yaml", "generator.pt", "tokenizer.pt"):
    path = hf_hub_download(
        repo_id="hazyresearch/ncm-checkpoints",
        filename=f"xsum/{filename}",
        local_dir=CHECKPOINT_DIR.parent,
    )
    downloaded = Path(path)
    if downloaded.parent != CHECKPOINT_DIR:
        downloaded.replace(CHECKPOINT_DIR / filename)

print("Checkpoints:", CHECKPOINT_DIR)
print([p.name for p in CHECKPOINT_DIR.iterdir()])

config.yaml:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

xsum/generator.pt: reconstructing file:   0%|          |  0.00B /  354MB            

xsum/generator.pt: downloading bytes:           |  0.00B            

xsum/tokenizer.pt: reconstructing file:   0%|          |  0.00B /  300MB            

xsum/tokenizer.pt: downloading bytes:           |  0.00B            

Checkpoints: /workspace/models/ncm-checkpoints/xsum
['tokenizer.pt', 'config.yaml', 'generator.pt']


In [5]:
from omegaconf import OmegaConf

CONFIG_PATH = CHECKPOINT_DIR / "config.yaml"
GENERATOR_PATH = CHECKPOINT_DIR / "generator.pt"
TOKENIZER_PATH = CHECKPOINT_DIR / "tokenizer.pt"

CFG = OmegaConf.load(CONFIG_PATH)
CFG.tokenizer.vqvae.checkpoint_path = str(TOKENIZER_PATH)

from tokenization.vqvae_tokenizer import VQVAETokenizer
from models.next_concept import NextConceptTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Este experimento precisa de GPU NVIDIA no container.")

tk = VQVAETokenizer.from_config(CFG.tokenizer)
tk.vqvae.to(device)
tk.vqvae.eval()
for param in tk.vqvae.parameters():
    param.requires_grad_(False)

net = NextConceptTransformer(vqvae_local=tk.vqvae, config=CFG.model).to(device)
checkpoint = torch.load(GENERATOR_PATH, map_location="cpu", weights_only=False)
net.load_state_dict(checkpoint["model"], strict=True)
net.eval()

trainable = [p for p in net.parameters() if p.requires_grad]
print("Gerador total:", round(sum(p.numel() for p in net.parameters()) / 1e6, 2), "M")
print("Parametros treinaveis:", round(sum(p.numel() for p in trainable) / 1e6, 2), "M")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

VQVAE, Total parameter count: 60.63M
VQVAE, Non embedding parameter count: 41.30M
NCP Transformer, Total parameter count: 88.02M
NCP Transformer, Non embedding parameter count: 87.71M
Gerador total: 88.02 M
Parametros treinaveis: 88.02 M


## Preparacao do MBPP

Usamos somente exemplos que cabem no adaptador seguro: prompt com ate 512 tokens GPT-2 e codigo com ate 48 tokens GPT-2. Isso evita entradas maiores que o contexto do checkpoint sem mascarar o filtro.

In [7]:
from datasets import load_dataset

mbpp = load_dataset("google-research-datasets/mbpp", "sanitized")
PAD_ID = int(CFG.model.prefix_pad_token_id)

def valid_example(item):
    prompt_ids = tk.pre_tokenizer.encode(item["prompt"])
    code_ids = tk.pre_tokenizer.encode(item["code"])
    return len(prompt_ids) <= 512 and 1 <= len(code_ids) <= 48

train_items = [item for item in mbpp["train"] if valid_example(item)]
test_items = [item for item in mbpp["test"] if valid_example(item)]
print("treino filtrado:", len(train_items), "/", len(mbpp["train"]))
print("teste filtrado:", len(test_items), "/", len(mbpp["test"]))
print("exemplo:", train_items[0]["prompt"][:120])

treino filtrado: 52 / 120
teste filtrado: 108 / 257
exemplo: Write a function to reverse words seperated by spaces in a given string.


In [9]:
@torch.no_grad()
def make_training_batch(items):
    prefix_chunks = []
    ncm_chunks = []
    target_chunks = []
    prefix_mask_chunks = []

    for item in items:
        prompt_ids = tk.pre_tokenizer.encode(item["prompt"])
        prompt_ids = prompt_ids[:512]
        prompt_ids = prompt_ids + [PAD_ID] * (512 - len(prompt_ids))
        prefix = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
        prefix_mask_chunks.append(prefix.eq(PAD_ID))

        code_ids = tk.pre_tokenizer.encode(item["code"])
        code_ids = code_ids[:48]
        code_ids = code_ids + [tk.pre_tokenizer.pad_token_id] * (48 - len(code_ids))
        code_tensor = torch.tensor(code_ids, dtype=torch.long, device=device).unsqueeze(0)

        target = tk.vqvae.encode(code_tensor)
        ncm_input = tk.vqvae.vq.idx_to_ncm_input(target)
        if net.query_level0:
            l0_query = net.build_l0_query_input(1, dtype=ncm_input.dtype)
            ncm_input = torch.cat([l0_query, ncm_input], dim=1)
        prefix_embedding = tk.vqvae.encoder.wte(prefix)
        ncm_chunks.append(ncm_input)
        prefix_chunks.append(prefix_embedding)
        target_chunks.append(target)

    prefix_embedding = torch.cat(prefix_chunks, dim=0)
    ncm_input = torch.cat(ncm_chunks, dim=0)
    model_input = torch.cat([prefix_embedding, ncm_input], dim=1)
    targets = torch.cat(target_chunks, dim=0)
    prefix_pad_mask = torch.cat(prefix_mask_chunks, dim=0)
    return model_input, targets, prefix_pad_mask

smoke_items = train_items[:1]
smoke_input, smoke_targets, smoke_prefix_pad_mask = make_training_batch(smoke_items)
net.eval()
with torch.inference_mode():
    smoke_logits = net.forward(smoke_input, prefix_pad_mask=smoke_prefix_pad_mask)

print("entrada:", tuple(smoke_input.shape))
print("alvos:", tuple(smoke_targets.shape))
print("logits:", tuple(smoke_logits.shape))
assert smoke_input.shape[1] == net.prefix_len + sum(net.levels)
assert smoke_logits.shape[:2] == smoke_targets.shape
print("Smoke test: OK")

entrada: (1, 712, 384)
alvos: (1, 200)
logits: (1, 200, 4096)
Smoke test: OK


## Treino conservador

A celula abaixo so muda pesos do gerador. O VQ-VAE continua congelado. Checkpoints sao gravados em `/workspace/results/ncm_codigo`.

In [10]:
import matplotlib.pyplot as plt
from torch.nn import functional as F

RUN_TRAINING = True
BATCH_SIZE = 1
ACCUMULATION_STEPS = 8
LEARNING_RATE = 5e-5
EPOCHS = 2
MAX_TRAIN_ITEMS = 128

OUTPUT_DIR = Path("/workspace/results/ncm_codigo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

loss_history = []

if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Treino interrompido: CUDA indisponivel.")

    net.train()
    net.vqvae_local[0].eval()
    optimizer = torch.optim.AdamW(
        (p for p in net.parameters() if p.requires_grad),
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=True)
    optimizer.zero_grad(set_to_none=True)

    selected = train_items[:MAX_TRAIN_ITEMS]
    for epoch in range(EPOCHS):
        random.shuffle(selected)
        epoch_loss = 0.0
        steps = 0

        for start in range(0, len(selected), BATCH_SIZE):
            batch_items = selected[start:start + BATCH_SIZE]
            model_input, targets = make_training_batch(batch_items)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = net.forward(model_input)
                loss = F.cross_entropy(
                    logits.float().reshape(-1, logits.shape[-1]),
                    targets.reshape(-1),
                ) / ACCUMULATION_STEPS

            scaler.scale(loss).backward()
            steps += 1

            if steps % ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * ACCUMULATION_STEPS
            if steps % 8 == 0:
                print(f"epoca={epoch + 1} passo={steps} loss={epoch_loss / steps:.4f}")

        mean_loss = epoch_loss / max(steps, 1)
        loss_history.append(mean_loss)
        checkpoint_path = OUTPUT_DIR / f"epoch_{epoch + 1}.pt"
        torch.save({
            "model": net.state_dict(),
            "epoch": epoch + 1,
            "loss": mean_loss,
            "config": OmegaConf.to_container(CFG, resolve=True),
        }, checkpoint_path)
        print("checkpoint:", checkpoint_path)

    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
    plt.xlabel("Epoca")
    plt.ylabel("Cross entropy")
    plt.title("Perda de treino NCM - Python")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "loss_curve.png")
    plt.show()
else:
    print("Treino preservado desativado. Ajuste RUN_TRAINING=True somente depois do smoke test.")

Treino preservado desativado. Ajuste RUN_TRAINING=True somente depois do smoke test.


## Geracao e validacao sintetica

A geração usa o checkpoint atual, seja o checkpoint oficial ou um checkpoint salvo após treino. O resultado pode ser sintaticamente inválido porque o modelo não foi treinado em código.

In [11]:
if test_items:
    sample = test_items[0]
else:
    sample = train_items[0]

prefix_ids = tk.pre_tokenizer.encode(sample["prompt"])
prefix_ids = prefix_ids[:512]
prefix_ids = prefix_ids + [PAD_ID] * (512 - len(prefix_ids))
prefix_tensor = torch.tensor(prefix_ids, dtype=torch.long, device=device).unsqueeze(0)

net.eval()
with torch.inference_mode():
    concept_ids = net.generate(
        idx=None,
        prefix_tokens=prefix_tensor,
        temperature=0.8,
        top_k=40,
        top_p=0.9,
    )
    logits = tk.vqvae.decode(concept_ids)
    raw_ids = logits.squeeze(0).argmax(dim=-1)
    generated = tk.pre_tokenizer.decode(raw_ids.tolist(), skip_special_tokens=True)

print("prompt:", sample["prompt"])
print("codigo gerado:\n", generated)
try:
    ast.parse(generated)
    print("Sintaxe Python: valida")
except SyntaxError as exc:
    print("Sintaxe Python: invalida -", exc)

prompt: Write a function to sort a given matrix in ascending order according to the sum of its rows.
codigo gerado:
 A smartphone that can detect a function,, function or ratifiedifying - or or a or being used in a tiny form of metal concrete.. Last in
Sintaxe Python: invalida - invalid syntax (<unknown>, line 1)


## Limitacoes

- o checkpoint XSum nao foi treinado para indentacao ou sintaxe Python;
+- MBPP pequeno e a possivel adaptacao nao garante codigo executavel;
+- o notebook congela o tokenizer, entao o XSum pode nao representar caracteres Python de maneira adequada;
+- esta primeira versao testa o fluxo e serve como baseline experimental;
+- nao use o checkpoint como modelo de producao sem validacao humana e testes executaveis.